In [90]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [91]:
df = pd.read_csv('/run/media/awongo-fahadi-rashid/AFRAH/Projects/Recess Term/Lectures/Datasets/Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [92]:
# Create a copy of the dataframe to clean and preprocess
cleaned = df.replace(' ', np.nan)
print(cleaned.isnull().sum())

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


In [93]:
# Convert TotalCharges to float, forcing any remaining weird characters to NaN
cleaned['TotalCharges'] = pd.to_numeric(cleaned['TotalCharges'], errors='coerce')

In [94]:
# To use Median:
median_value = cleaned['TotalCharges'].median()
cleaned['TotalCharges'] = cleaned['TotalCharges'].fillna(median_value)

# To use Mean:
# mean_value = cleaned['TotalCharges'].mean()
# cleaned['TotalCharges'] = cleaned['TotalCharges'].fillna(mean_value)

In [95]:
cleaned.isnull().sum()  # Check again for missing values

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [96]:
# Check for categorical variables and their unique values
categorical_cols = cleaned.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"{col}: {cleaned[col].nunique()} unique values")

customerID: 7043 unique values
gender: 2 unique values
Partner: 2 unique values
Dependents: 2 unique values
PhoneService: 2 unique values
MultipleLines: 3 unique values
InternetService: 3 unique values
OnlineSecurity: 3 unique values
OnlineBackup: 3 unique values
DeviceProtection: 3 unique values
TechSupport: 3 unique values
StreamingTV: 3 unique values
StreamingMovies: 3 unique values
Contract: 3 unique values
PaperlessBilling: 2 unique values
PaymentMethod: 4 unique values
Churn: 2 unique values


/tmp/ipykernel_20667/2155924924.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = cleaned.select_dtypes(include=['object']).columns


In [97]:
from sklearn.preprocessing import LabelEncoder

# 1. Define column groups
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
multi_class_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

# 2. Apply Label Encoding to binary categorical variables
for col in binary_cols:
    encoder = LabelEncoder()
    cleaned[col] = pd.Series(encoder.fit_transform(cleaned[col].astype(str)), index=cleaned.index)

# 3. Apply One-Hot Encoding to multi-class categorical variables (In one go!)
cleaned = pd.get_dummies(cleaned, columns=multi_class_cols, drop_first=True)

# 4. View results
cleaned.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,7590-VHVEG,0,0,1,0,1,0,1,29.85,29.85,...,False,False,False,False,False,False,False,False,True,False
1,5575-GNVDE,1,0,0,0,34,1,0,56.95,1889.50,...,False,False,False,False,False,True,False,False,False,True
2,3668-QPYBK,1,0,0,0,2,1,1,53.85,108.15,...,False,False,False,False,False,False,False,False,False,True
3,7795-CFOCW,1,0,0,0,45,0,0,42.30,1840.75,...,True,False,False,False,False,True,False,False,False,False
4,9237-HQITU,0,0,0,0,2,1,1,70.70,151.65,...,False,False,False,False,False,False,False,False,True,False


In [98]:
from sklearn.model_selection import train_test_split
X = cleaned.drop(['customerID', 'Churn'], axis=1)
y = cleaned['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
cleaned.shape, X_train.shape, X_test.shape, y_train.shape, y_test.shape

((7043, 32), (5634, 30), (1409, 30), (5634,), (1409,))

In [99]:
# Feature Scaling: Used to standardize the range of independent variables or features of data. 
# Standardization of a dataset is a common requirement for many machine learning estimators: 
# they might behave badly if the individual features do not more or less look like standard 
# normally distributed data (e.g., Gaussian with 0 mean and unit variance).
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [101]:
# Ensemble Learning: A machine learning paradigm where multiple models (often called 
# "weak learners") are trained to solve the same problem and combined to get better results. 
# The main idea is that by combining multiple models, the ensemble can achieve better 
# performance than any individual model.
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier


models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=0),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=0),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=0)
}
metrics = []
for model in models:
    models[model].fit(X_train_scaled, y_train)
    print(f"{model} trained.")
# Get the metrics for each model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
for model_name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    metrics.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })
# Print a formatted table of the metrics
metrics_df = pd.DataFrame(metrics)
print(metrics_df)

Random Forest trained.
Gradient Boosting trained.
AdaBoost trained.
               Model  Accuracy  Precision    Recall  F1 Score
0      Random Forest  0.781405   0.605634  0.467391  0.527607
1  Gradient Boosting  0.790632   0.625430  0.494565  0.552352
2           AdaBoost  0.792761   0.634752  0.486413  0.550769
